## Three ways to consume a ConfigMap

A ConfigMap does nothing alone. A pod *uses* it in one (or more) of three ways.

**1. As individual env vars — `configMapKeyRef`.** Pull one key into one env var. Most precise.

```yaml
env:
  - name: LOG_LEVEL
    valueFrom:
      configMapKeyRef: { name: app-config, key: LOG_LEVEL }
```

Use it to rename a key or take only a subset.

**2. As a bulk env import — `envFrom`.** Slurp *every* key/value into env vars, key = variable name.

```yaml
envFrom:
  - configMapRef: { name: app-config }
    prefix: APP_       # optional — LOG_LEVEL → APP_LOG_LEVEL
```

Use it when the keys already *are* your env-var names.

**3. As files in a volume — `volume.configMap`.** Mount as a directory; each key becomes a file whose contents are the value.

```yaml
volumeMounts: [{ name: cfg, mountPath: /etc/app }]
volumes:
  - name: cfg
    configMap:
      name: app-file
      items: [{ key: app.properties, path: app.properties, mode: 0644 }]
```

Right when the app reads a *file* (JVM apps, `--config /path`).

| App reads… | Use |
|---|---|
| a few specific env vars | `configMapKeyRef` |
| many env vars from one source | `envFrom` |
| a config file | volume mount |
| config that live-updates | volume mount (next) |

On our map this is the arrow from **ConfigMap** to **env / mount** — the same data, three projections into the Pod.